# Homework: Vector Search

In this homework, we put what we learned in Module 2 into practice.
We use the ONNX `Embedder` for lightweight embeddings.

In [1]:
import numpy as np
from embedder import Embedder

embed = Embedder()

2026-06-28 11:22:30.305389587 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


## Q1. Embedding a query

Embed the following query:
> How does approximate nearest neighbor search work?

What's the first value (`v[0]`)?

In [2]:
query = "How does approximate nearest neighbor search work?"
v = embed.encode(query)
print(f"v[0] = {v[0]:.2f}")
print(f"Shape: {v.shape}")

v[0] = -0.02
Shape: (384,)


**A1.** `v[0] = -0.02`

## Loading the data

Pull lesson pages from the course repository (commit `8c1834d`, 72 pages).

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
print(f"Number of documents: {len(documents)}")

Number of documents: 72


## Q2. Cosine similarity

Embed the content of page `02-vector-search/lessons/07-sqlitesearch-vector.md` and compute cosine similarity with the query vector from Q1.

In [4]:
target_filename = "02-vector-search/lessons/07-sqlitesearch-vector.md"
page = next(doc for doc in documents if doc["filename"] == target_filename)

page_vector = embed.encode(page["content"])
cosine_sim = v.dot(page_vector)
print(f"Cosine similarity: {cosine_sim:.2f}")

Cosine similarity: 0.36


## Q3. Chunking and search by hand

Chunk the pages, embed every chunk's `content`, and find the highest-scoring chunk for the Q1 query.

In [5]:
from gitsource import chunk_documents
from tqdm.auto import tqdm

chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Number of chunks: {len(chunks)}")

Number of chunks: 295


In [6]:
# Embed all chunk contents in batches
texts = [chunk["content"] for chunk in chunks]
batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)
print(f"Embedding matrix shape: {X.shape}")

  0%|          | 0/6 [00:00<?, ?it/s]

Embedding matrix shape: (295, 384)


In [7]:
scores = X.dot(v)
best_idx = scores.argmax()
print(f"Highest-scoring chunk filename: {chunks[best_idx]['filename']}")
print(f"Score: {scores[best_idx]:.4f}")

Highest-scoring chunk filename: 02-vector-search/lessons/07-sqlitesearch-vector.md
Score: 0.6489


## Q4. Vector search with minsearch

Use `VectorSearch` from minsearch to search for:
> What metric do we use to evaluate a search engine?

In [8]:
from minsearch import VectorSearch

vector_index = VectorSearch()
vector_index.fit(X, chunks)

In [9]:
q4_query = "What metric do we use to evaluate a search engine?"
q4_vector = embed.encode(q4_query)

q4_results = vector_index.search(
    query_vector=q4_vector,
    num_results=5,
)

print(f"First result filename: {q4_results[0]['filename']}")
for i, r in enumerate(q4_results):
    print(f"  {i+1}. {r['filename']}")

First result filename: 04-evaluation/lessons/05-search-metrics.md
  1. 04-evaluation/lessons/05-search-metrics.md
  2. 04-evaluation/lessons/01-intro.md
  3. 01-agentic-rag/lessons/05-search.md
  4. 04-evaluation/lessons/01-intro.md
  5. 04-evaluation/lessons/15-next-steps.md


## Q5. Text search vs vector search

Compare vector and text search for:
> How do I store vectors in PostgreSQL?

Which file shows up in vector results but not in text results (top 5)?

In [10]:
from minsearch import Index

text_index = Index(text_fields=["content"])
text_index.fit(chunks)

In [11]:
q5_query = "How do I store vectors in PostgreSQL?"
q5_vector = embed.encode(q5_query)

vector_results = vector_index.search(
    query_vector=q5_vector,
    num_results=5,
)

text_results = text_index.search(query=q5_query, num_results=5)

vector_files = {r["filename"] for r in vector_results}
text_files = {r["filename"] for r in text_results}

only_in_vector = vector_files - text_files

print("Vector search top 5:")
for i, r in enumerate(vector_results):
    print(f"  {i+1}. {r['filename']}")

print("\nText search top 5:")
for i, r in enumerate(text_results):
    print(f"  {i+1}. {r['filename']}")

print(f"\nIn vector but not in text: {only_in_vector}")

Vector search top 5:
  1. 02-vector-search/lessons/08-pgvector.md
  2. 02-vector-search/lessons/08-pgvector.md
  3. 03-orchestration/lessons/05-rag.md
  4. 02-vector-search/lessons/08-pgvector.md
  5. 02-vector-search/lessons/08-pgvector.md

Text search top 5:
  1. 02-vector-search/lessons/02-embeddings.md
  2. 03-orchestration/lessons/05-rag.md
  3. 02-vector-search/lessons/01-intro.md
  4. 03-orchestration/lessons/05-rag.md
  5. 02-vector-search/lessons/01-intro.md

In vector but not in text: {'02-vector-search/lessons/08-pgvector.md'}
